In [1]:
import os
os.chdir("C:/Users/Emanuel Osorio/Repositorio DS/Libros_Electiva") 
print(os.listdir())

['artifacts', 'data', 'docs', 'Notebook_1.ipynb', 'src']


In [2]:
# 1) Setup (cross-platform)
import sys, platform, hashlib
from pathlib import Path
import duckdb
import os

os.chdir("..")

PROJECT_ROOT = Path(".").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR  = DATA_DIR / "raw"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
DOCS_DIR = PROJECT_ROOT / "docs"

for d in [RAW_DIR, SILVER_DIR, GOLD_DIR, ARTIFACTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DIR / "exoplanets.duckdb"
con = duckdb.connect(str(DB_PATH))

def show(path: Path):
    return {"exists": path.exists(), "path": str(path), "size_bytes": path.stat().st_size if path.exists() else None}


print("OS:", platform.platform())
print("Python:", sys.version.split()[0])
print("DuckDB:", con.execute("SELECT version()").fetchone()[0])
show(DB_PATH)

OS: Windows-11-10.0.26200-SP0
Python: 3.13.2
DuckDB: v1.4.4


{'exists': True,
 'path': 'C:\\Users\\Emanuel Osorio\\Repositorio DS\\data\\exoplanets.duckdb',
 'size_bytes': 12288}

In [3]:
con.execute("SELECT 42 AS answer").fetchall()

[(42,)]

In [4]:
con.execute("CREATE OR REPLACE TABLE demo_numbers(x INTEGER)")
con.execute("INSERT INTO demo_numbers VALUES (1), (2), (3)")
con.execute("SELECT SUM(x) AS s FROM demo_numbers").fetchall()

[(6,)]

In [5]:
# TU TURNO 1
con.execute("CREATE OR REPLACE TABLE students(name VARCHAR, semester INTEGER)")
con.execute("INSERT INTO students VALUES ('Ana', 7), ('Luis', 8), ('Sofia', 10)")
con.execute("SELECT semester, COUNT(*) n FROM students GROUP BY 1 ORDER BY 1").fetchall()

[(7, 1), (8, 1), (10, 1)]

In [6]:
# TU TURNO 2
con.execute("CREATE OR REPLACE TABLE submissions(name VARCHAR, lab VARCHAR, ok BOOLEAN)")
con.execute("""
INSERT INTO submissions VALUES
  ('Ana', 'W01', TRUE),
  ('Ana', 'W02', FALSE),
  ('Luis','W01', TRUE),
  ('Luis','W02', TRUE),
  ('Sofia','W01', TRUE)
""")
con.execute("SELECT name, COUNT(*) n FROM submissions GROUP BY 1 ORDER BY n DESC").fetchall()

[('Ana', 2), ('Luis', 2), ('Sofia', 1)]

In [7]:
try:
    con.close()
    print("DuckDB connection closed.")
except NameError:
    print("No connection named 'con' in this notebook.")

DuckDB connection closed.


In [17]:
# Setup común (cross-platform) + detección de raíz del proyecto
import sys, time, json, hashlib, platform, subprocess, os
from pathlib import Path
import duckdb

def find_project_root(start: Path) -> Path:
    """Busca hacia arriba una carpeta que contenga `src/` y `data/`.
    Esto evita errores cuando el notebook se ejecuta desde notebooks/."""
    cur = start.resolve()
    for p in [cur] + list(cur.parents):
        if (p / "src").exists() and (p / "data").exists():
            return p
    # search in subdirs
    for sub in cur.iterdir():
        if sub.is_dir() and (sub / "src").exists() and (sub / "data").exists():
            return sub
    # fallback: carpeta actual
    return cur

PROJECT_ROOT = Path("Libros_Electiva").resolve()
print("PROJECT_ROOT:", PROJECT_ROOT)

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
DOCS_DIR = PROJECT_ROOT / "docs"
for d in [RAW_DIR, ARTIFACTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DIR / "exoplanets.duckdb"
con = duckdb.connect(str(DB_PATH))

def show(path: Path):
    return {"exists": path.exists(), "path": str(path), "size_bytes": path.stat().st_size if path.exists() else None}

def sha256_file(path: Path, chunk_size: int = 1<<20) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def run_module(mod: str, *args: str):
        """Ejecuta `python src/...` desde la raíz del proyecto."""
        script_path = mod.replace(".", "/") + ".py"
        env = os.environ.copy()
        env["PYTHONPATH"] = str(PROJECT_ROOT)
        cmd = [sys.executable, script_path, *args]
        print("Running:", " ".join(cmd))
        result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), env=env, capture_output=True, text=True)
        if result.returncode != 0:
            print("STDOUT:", result.stdout)
            print("STDERR:", result.stderr)
            raise subprocess.CalledProcessError(result.returncode, cmd)

print("OS:", platform.platform())
print("Python:", sys.version.split()[0])
print("DuckDB:", con.execute("SELECT version()").fetchone()[0])

PROJECT_ROOT: C:\Users\Emanuel Osorio\Repositorio DS\Libros_Electiva
OS: Windows-11-10.0.26200-SP0
Python: 3.13.2
DuckDB: v1.4.4


In [12]:
EXPECTED = "pscomppars.csv"
raw_csv = RAW_DIR / EXPECTED

# Si no está con el nombre esperado, buscamos candidatos (para no bloquear la clase)
if not raw_csv.exists():
    candidates = list(RAW_DIR.glob("pscomppars*.csv")) + list(RAW_DIR.glob("*.csv"))
    candidates = [c for c in candidates if c.is_file()]
    if candidates:
        raw_csv = sorted(candidates)[0]
        print(f"No encontré {EXPECTED}. Encontré y usaré: {raw_csv.name}")
        print("   Recomendación: renombra a pscomppars.csv para estandarizar el curso.")
    else:
        raw_csv = None

raw_csv

In [10]:
print("sys.executable:", sys.executable)

sys.executable: c:\Users\Emanuel Osorio\Repositorio DS\.venv\Scripts\python.exe


In [18]:
# Si no hay Raw, NO intentamos descargar automáticamente (para evitar fallos sin internet).
# Si quieres descargar desde clase (y tienes internet), pon DO_DOWNLOAD=True.
DO_DOWNLOAD = True

if raw_csv is None:
    if DO_DOWNLOAD:
        # Descarga oficial del curso (requiere internet). Escribe en data/raw/pscomppars.csv
        run_module("src.ingest.download_exoplanets", "--format", "csv", "--limit", "50000")
        raw_csv = RAW_DIR / "pscomppars.csv"
    else:
        raise FileNotFoundError(
            "No encontré ningún CSV en data/raw/.\n"
            "Solución A (recomendada): coloca pscomppars.csv en data/raw/.\n"
            "Solución B (con internet): cambia DO_DOWNLOAD=True y re-ejecuta esta celda."
        )

# Evidencia de trazabilidad
show(raw_csv), sha256_file(raw_csv)

Running: c:\Users\Emanuel Osorio\Repositorio DS\.venv\Scripts\python.exe src/ingest/download_exoplanets.py --format csv --limit 50000


({'exists': True,
  'path': 'C:\\Users\\Emanuel Osorio\\Repositorio DS\\Libros_Electiva\\data\\raw\\pscomppars.csv',
  'size_bytes': 927404},
 'ecf89df899d539fe0a7e73ee35449192733ced7b83e6afdc2f9357a9121bd631')

In [ ]:
path = str(raw_csv).replace("\\", "/").replace("'", "''")  # Windows-safe
con.execute(f"""
CREATE OR REPLACE VIEW raw_ps AS
SELECT * FROM read_csv_auto('{path}')
""")
